In [1]:
# ============================================================
# SENTIMENT ANALYSIS SYSTEM
# Cell 1 - Install Required Libraries
# ============================================================

!pip install -q transformers torch gradio pandas matplotlib

print("✅ All required libraries installed successfully!")
print("🚀 Sentiment Analysis System setup completed.")

✅ All required libraries installed successfully!
🚀 Sentiment Analysis System setup completed.


In [2]:
# ============================================================
# SENTIMENT ANALYSIS SYSTEM
# Cell 2 - Load Pre-trained AI Model
# ============================================================

from transformers import pipeline
import torch

# Select device automatically
device = 0 if torch.cuda.is_available() else -1

print("🔄 Loading AI Sentiment Analysis Model...")

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device
)

print("✅ AI model loaded successfully!")
print("🤖 Sentiment Analysis Engine is ready.")

🔄 Loading AI Sentiment Analysis Model...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ AI model loaded successfully!
🤖 Sentiment Analysis Engine is ready.


In [3]:
# ============================================================
# SENTIMENT ANALYSIS SYSTEM
# Cell 3 - Sentiment Analysis Engine
# ============================================================

import pandas as pd

analysis_history = []


def analyze_sentiment(text):
    """
    Analyze the sentiment of the given text
    and return sentiment, confidence and details.
    """

    if not text or not text.strip():
        return {
            "sentiment": "No Input",
            "confidence": 0.0,
            "message": "Please enter some text for analysis."
        }

    text = text.strip()

    try:
        result = sentiment_model(text)[0]

        label = result["label"].lower()
        confidence = result["score"] * 100

        # Convert model labels into readable names
        if "positive" in label or label == "label_2":
            sentiment = "Positive"
            emoji = "😊"
        elif "negative" in label or label == "label_0":
            sentiment = "Negative"
            emoji = "😞"
        else:
            sentiment = "Neutral"
            emoji = "😐"

        # Save result to history
        analysis_history.append({
            "Text": text,
            "Sentiment": sentiment,
            "Confidence": round(confidence, 2)
        })

        return {
            "sentiment": sentiment,
            "confidence": confidence,
            "message": f"{emoji} {sentiment} sentiment detected with {confidence:.2f}% confidence."
        }

    except Exception as error:
        return {
            "sentiment": "Error",
            "confidence": 0.0,
            "message": f"Analysis failed: {str(error)}"
        }


# Test the engine
test_result = analyze_sentiment(
    "I really enjoyed this product. It was excellent!"
)

print("🧪 Test Result")
print("------------------------------")
print("Sentiment :", test_result["sentiment"])
print("Confidence:", f"{test_result['confidence']:.2f}%")
print("Message   :", test_result["message"])

🧪 Test Result
------------------------------
Sentiment : Positive
Confidence: 98.88%
Message   : 😊 Positive sentiment detected with 98.88% confidence.


In [4]:
# ============================================================
# SENTIMENT ANALYSIS SYSTEM
# Cell 4 - Professional User Interface
# ============================================================

import gradio as gr

custom_css = """
body {
    background: #0f172a !important;
}

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    background: #0f172a !important;
    font-family: Arial, sans-serif !important;
}

.hero {
    text-align: center;
    padding: 38px 20px 28px;
}

.hero h1 {
    color: white !important;
    font-size: 42px !important;
    margin-bottom: 8px !important;
}

.hero p {
    color: #94a3b8 !important;
    font-size: 16px !important;
    line-height: 1.6;
}

.card {
    background: #172033 !important;
    border: 1px solid #273653 !important;
    border-radius: 20px !important;
    padding: 24px !important;
}

.section-title {
    color: white !important;
    font-size: 20px !important;
    font-weight: 700 !important;
    margin-bottom: 15px !important;
}

.result-box {
    border-radius: 16px;
    padding: 22px;
    text-align: center;
    background: #111827;
    border: 1px solid #334155;
}

.result-title {
    color: #94a3b8;
    font-size: 14px;
    margin-bottom: 8px;
}

.result-value {
    color: white;
    font-size: 30px;
    font-weight: bold;
}

.confidence {
    color: #cbd5e1;
    font-size: 15px;
    margin-top: 8px;
}

.feature {
    background: #111827;
    border: 1px solid #273653;
    border-radius: 14px;
    padding: 18px;
    text-align: center;
    color: #cbd5e1;
    min-height: 90px;
}

.footer {
    text-align: center;
    color: #64748b;
    padding: 28px 10px;
    font-size: 13px;
}

button {
    border-radius: 12px !important;
    font-weight: 600 !important;
}

#analyze-btn {
    height: 52px !important;
    font-size: 16px !important;
}
"""


def analyze_for_ui(text):
    result = analyze_sentiment(text)

    sentiment = result["sentiment"]
    confidence = result["confidence"]
    message = result["message"]

    return (
        message,
        sentiment,
        f"{confidence:.2f}%",
        len(text.split()) if text else 0
    )


with gr.Blocks(
    title="Sentiment Intelligence",
    css=custom_css,
    theme=gr.themes.Base()
) as app:

    # --------------------------------------------------------
    # HERO
    # --------------------------------------------------------

    gr.HTML("""
    <div class="hero">
        <h1>🧠 Sentiment Intelligence</h1>
        <p>
            AI-powered sentiment analysis for understanding
            <br>
            emotions and opinions hidden inside text.
        </p>
    </div>
    """)

    # --------------------------------------------------------
    # MAIN WORKSPACE
    # --------------------------------------------------------

    with gr.Row():

        # INPUT
        with gr.Column(scale=1, elem_classes="card"):

            gr.HTML("""
            <div class="section-title">
                ✍️ Text Analysis
            </div>
            """)

            text_input = gr.Textbox(
                label="Enter Text or Review",
                placeholder=(
                    "Example: I really loved this product. "
                    "The quality was amazing!"
                ),
                lines=10
            )

            with gr.Row():

                analyze_btn = gr.Button(
                    "✨ Analyze Sentiment",
                    variant="primary",
                    elem_id="analyze-btn"
                )

                clear_btn = gr.Button(
                    "🗑️ Clear",
                    variant="secondary"
                )

        # RESULT
        with gr.Column(scale=1, elem_classes="card"):

            gr.HTML("""
            <div class="section-title">
                📊 Analysis Result
            </div>
            """)

            result_message = gr.Textbox(
                label="AI Interpretation",
                interactive=False,
                lines=3
            )

            with gr.Row():

                sentiment_output = gr.Textbox(
                    label="Detected Sentiment",
                    value="—",
                    interactive=False
                )

                confidence_output = gr.Textbox(
                    label="Confidence",
                    value="0%",
                    interactive=False
                )

            word_count_output = gr.Number(
                label="Word Count",
                value=0,
                precision=0,
                interactive=False
            )

    # --------------------------------------------------------
    # FEATURES
    # --------------------------------------------------------

    gr.Markdown("### ✨ System Capabilities")

    with gr.Row():

        gr.HTML("""
        <div class="feature">
            <b>🤖 AI Powered</b><br>
            Transformer-based NLP analysis
        </div>
        """)

        gr.HTML("""
        <div class="feature">
            <b>😊 Emotion Insight</b><br>
            Positive, Neutral & Negative
        </div>
        """)

        gr.HTML("""
        <div class="feature">
            <b>📊 Confidence</b><br>
            Model confidence percentage
        </div>
        """)

        gr.HTML("""
        <div class="feature">
            <b>⚡ Fast Analysis</b><br>
            Instant text processing
        </div>
        """)

    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="footer">
        Sentiment Intelligence System
        <br>
        Built with Python • Transformers • PyTorch • Gradio
    </div>
    """)

    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    analyze_btn.click(
        fn=analyze_for_ui,
        inputs=text_input,
        outputs=[
            result_message,
            sentiment_output,
            confidence_output,
            word_count_output
        ]
    )

    clear_btn.click(
        fn=lambda: ("", "", "0%", 0),
        inputs=None,
        outputs=[
            text_input,
            result_message,
            confidence_output,
            word_count_output
        ]
    )

app.launch(share=True)

/tmp/ipykernel_2131/410848959.py:121: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://147922fab78d801546.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
